In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.anova import anova_lm

df = pd.read_csv("social_media_dataset.csv")


df['engagement'] = df['likes'] + df['shares'] + df['comments_count']


df['conversion_intent'] = df['engagement'] / df['views']

categories = df['content_category'].unique()

for cat in categories:

    print("\n==============================")
    print("Influencer Category:", cat)
    print("==============================")

    df_cat = df[df['content_category'] == cat].dropna()

    df_cat['log_intent'] = np.log1p(df_cat['conversion_intent'])
    df_cat['log_followers'] = np.log1p(df_cat['follower_count'])
    df_cat['log_views'] = np.log1p(df_cat['views'])
    df_cat['log_engagement'] = np.log1p(df_cat['engagement'])

    y = df_cat['log_intent']

    # Model 1
    X1 = sm.add_constant(df_cat[['log_followers']])

    # Model 2 (Reach)
    X2 = sm.add_constant(df_cat[['log_followers','log_views']])

    # Model 3 (Reach + Engagement)
    X3 = sm.add_constant(df_cat[['log_followers',
                                'log_views',
                                'log_engagement']])

    m1 = sm.OLS(y,X1).fit()
    m2 = sm.OLS(y,X2).fit()
    m3 = sm.OLS(y,X3).fit()

    print("\nM1 vs M2 (Does Reach Improve?)")
    print(anova_lm(m1,m2))

    print("\nM2 vs M3 (Does Engagement Improve?)")
    print(anova_lm(m2,m3))


Influencer Category: beauty

M1 vs M2 (Does Reach Improve?)
   df_resid       ssr  df_diff   ss_diff            F  Pr(>F)
0   14446.0  0.236001      0.0       NaN          NaN     NaN
1   14445.0  0.196575      1.0  0.039425  2897.097257     0.0

M2 vs M3 (Does Engagement Improve?)
   df_resid       ssr  df_diff   ss_diff             F  Pr(>F)
0   14445.0  0.196575      0.0       NaN           NaN     NaN
1   14444.0  0.000050      1.0  0.196526  5.713442e+07     0.0

Influencer Category: lifestyle

M1 vs M2 (Does Reach Improve?)
   df_resid       ssr  df_diff   ss_diff            F  Pr(>F)
0   14497.0  0.239934      0.0       NaN          NaN     NaN
1   14496.0  0.202760      1.0  0.037174  2657.682819     0.0

M2 vs M3 (Does Engagement Improve?)
   df_resid      ssr  df_diff  ss_diff             F  Pr(>F)
0   14496.0  0.20276      0.0      NaN           NaN     NaN
1   14495.0  0.00005      1.0  0.20271  5.852771e+07     0.0

Influencer Category: tech

M1 vs M2 (Does Reach Improve?

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error



perf = pd.read_excel("influencer campaign performance.xlsx")
inf = pd.read_excel("influencers.xlsx")
camp = pd.read_excel("campaigns.xlsx")



merged = perf.merge(
    inf,
    left_on="Source_Influencer_ID",
    right_on="Influencer_ID",
    how="left"
)

merged = merged.merge(
    camp,
    on="Campaign_ID",
    how="left"
)



merged = merged[merged['Cost'] > 0]
merged['engagement'] = merged['Revenue_Generated'] / merged['Cost']



merged['Start_Date'] = pd.to_datetime(merged['Start_Date'])
merged['End_Date'] = pd.to_datetime(merged['End_Date'])

merged['duration'] = (merged['End_Date'] - merged['Start_Date']).dt.days



model_df = merged[['Follower_Count',
                   'engagement',
                   'duration',
                   'Conversions']].dropna()

model_df['log_sales'] = np.log1p(model_df['Conversions'])
model_df['log_reach'] = np.log1p(model_df['Follower_Count'])
model_df['log_eng'] = np.log1p(model_df['engagement'])
model_df['log_duration'] = np.log1p(model_df['duration'])

y = model_df['log_sales']


# BASE MODEL (Reach)


X_base = model_df[['log_reach']]

X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42)

lr_base = LinearRegression()
lr_base.fit(X_train, y_train)

y_pred = lr_base.predict(X_test)

print("BASE MODEL")
print("R²:", r2_score(y_test, y_pred))

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE:", rmse)


# ENHANCED MODEL

X_enh = model_df[['log_reach','log_eng','log_duration']]

X_train, X_test, y_train, y_test = train_test_split(
    X_enh, y, test_size=0.2, random_state=42)

lr_enh = LinearRegression()
lr_enh.fit(X_train, y_train)

y_pred = lr_enh.predict(X_test)

print("\nENHANCED MODEL")
print("R²:", r2_score(y_test, y_pred))

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE:", rmse)

BASE MODEL
R²: 0.7018063181275496
RMSE: 0.2591550778165075

ENHANCED MODEL
R²: 0.6915877837388346
RMSE: 0.2635580527111831


In [ ]:


import pandas as pd
import numpy as np
from scipy.stats import ttest_ind


perf = pd.read_excel("influencer campaign performance.xlsx")
inf = pd.read_excel("influencers.xlsx")
camp = pd.read_excel("campaigns.xlsx")


merged = perf.merge(
    inf,
    left_on="Source_Influencer_ID",
    right_on="Influencer_ID",
    how="left"
)

merged = merged.merge(
    camp,
    on="Campaign_ID",
    how="left"
)



merged = merged.dropna(subset=[
    'Follower_Count',
    'Conversions',
    'Revenue_Generated',
    'Audience_Location_Match_pct',
    'ROI',
    'Tier'
])

merged = merged[merged['Follower_Count'] > 0]




merged['conversion_rate'] = merged['Conversions'] / merged['Follower_Count']
merged['rev_per_follower'] = merged['Revenue_Generated'] / merged['Follower_Count']
merged['roi_eff'] = merged['ROI']
merged['aud_match'] = merged['Audience_Location_Match_pct']



metrics = [
    'conversion_rate',
    'roi_eff',
    'rev_per_follower',
    'aud_match'
]

print("\n========== MEAN COMPARISON BY TIER ==========")

for m in metrics:
    print("\n------------------------------")
    print("Metric:", m)
    print("------------------------------")
    print(merged.groupby('Tier')[m].mean())



micro = merged[merged['Tier'] == "Micro"]
macro = merged[merged['Tier'] == "Macro"]

print("\n========== MICRO vs MACRO STATISTICAL TEST ==========")

for m in metrics:

    micro_vals = micro[m].dropna()
    macro_vals = macro[m].dropna()

    if len(micro_vals) < 2 or len(macro_vals) < 2:
        print("\nNot enough data for:", m)
        continue

    t_stat, p_val = ttest_ind(
        micro_vals,
        macro_vals,
        equal_var=False
    )

    print("\n------------------------------")
    print("Metric:", m)
    print("------------------------------")
    print("Micro Mean:", micro_vals.mean())
    print("Macro Mean:", macro_vals.mean())
    print("p-value:", p_val)


========== MEAN COMPARISON BY TIER ==========

------------------------------
Metric: conversion_rate
------------------------------
Tier
Macro    0.003874
Mega     0.002335
Micro    0.011250
Nano     0.032649
Name: conversion_rate, dtype: float64

------------------------------
Metric: roi_eff
------------------------------
Tier
Macro     172.826653
Mega       94.581939
Micro     680.067250
Nano     1358.206250
Name: roi_eff, dtype: float64

------------------------------
Metric: rev_per_follower
------------------------------
Tier
Macro     13.863219
Mega       8.154129
Micro     37.940996
Nano     129.899756
Name: rev_per_follower, dtype: float64

------------------------------
Metric: aud_match
------------------------------
Tier
Macro    65.086667
Mega     63.100000
Micro    76.162500
Nano     65.125000
Name: aud_match, dtype: float64

========== MICRO vs MACRO STATISTICAL TEST ==========

------------------------------
Metric: conversion_rate
------------------------------
Micro